# Fundamentals 12 - Multi-Agentic System End-to-End

Objetivo: extender el recorrido single-agent del notebook 11 a una composicion con responsabilidades separadas, sin introducir Graph todavia.

Un solo `AgenticSystem` es owner de tres Agents:

```text
solver (required) -> judge (required) -> reviewer LM (optional)
                         |
                         +-> composed RunResult -> Lineage
```

Solver y judge forman la ruta obligatoria. El reviewer puede usar OpenAI, Bedrock o vLLM y degradarse sin convertir un resultado determinista correcto en error.


In [ ]:
import os
import agentic_systems as toolkit
PRETTY = False
scheduler = toolkit.scheduler(timeout_s=60, max_retries=0, max_tool_calls=8, max_turns=8)
local_runtime = toolkit.runtime(provider="python-runtime", model="python-runtime", region="local", scheduler=scheduler)
lm_runtime = toolkit.runtime(provider="auto", scheduler=scheduler)
lm_resolution = lm_runtime.describe()
force_local_only = bool(os.getenv("AGENTIC_SYSTEMS_FORCE_LOCAL_TUTORIALS"))
lm_available = lm_resolution["selected_provider"] != "auto" and not force_local_only
system = toolkit.system(runtime=local_runtime)
toolkit.show({
    "local_runtime": local_runtime.describe(),
    "lm_runtime": lm_resolution,
    "lm_available": lm_available,
    "force_local_only": force_local_only,
}, title="Runtime selection")


## Escenario didactico compartido

Se reutilizan el prompt, los numeros y el resultado esperado del notebook 11. La diferencia que se estudia es la division de responsabilidades entre Agents.


In [ ]:
USER_PROMPT = "Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2."
NUMBERS = [10, 20, 9, 4, 2]
EXPECTED = 42
toolkit.show({
    "prompt": USER_PROMPT,
    "numbers": NUMBERS,
    "expected": EXPECTED,
}, title="Shared arithmetic scenario")


## Parámetros de la integración multi-agent

| Parametro | Qué controla | Decision del notebook |
|---|---|---|
| `local_runtime` | Runtime de solver y judge. | `python-runtime` para ejecucion determinista. |
| `lm_runtime` | Runtime del reviewer opcional. | `provider="auto"` para conservar portabilidad. |
| `AGENTIC_SYSTEMS_PROVIDER_PRIORITY` | Prioridad OpenAI/Bedrock/vLLM. | Permite probar un backend sin cambiar Agents. |
| `AGENTIC_SYSTEMS_FORCE_LOCAL_TUTORIALS` | Omite el reviewer LM. | Mantiene CI y demos locales reproducibles. |
| `max_tool_calls=1` | Limite por Agent obligatorio. | Cada rol ejecuta exactamente una Tool. |
| `temperature=0.0` | Variabilidad. | Evita aleatoriedad en la ruta determinista. |
| `solve_contract` | Evidencia requerida del solver. | Exige `solve_arithmetic`. |
| `judge_contract` | Evidencia requerida del judge. | Exige `judge_result`. |
| `EXPECTED=42` | Criterio del judge. | Evita aceptar una respuesta sin verificacion. |


## 1) Declarar Tools por responsabilidad

`solve_arithmetic` produce el procedimiento y `judge_result` verifica el resultado. Separar Tools permite asignar contratos distintos a cada Agent.


In [ ]:
@toolkit.tool
def solve_arithmetic(numbers: list[int]) -> dict:
    a, b, c, d, e = numbers
    x1 = a + b
    x2 = x1 - c
    x3 = x2 * d
    x4 = x3 / e
    return {"procedure": [f"{a}+{b}={x1}", f"{x1}-{c}={x2}", f"{x2}*{d}={x3}", f"{x3}/{e}={x4:g}"], "result": int(x4) if float(x4).is_integer() else x4}
@toolkit.tool
def judge_result(result: float, expected: float) -> dict:
    ok = float(result) == float(expected)
    return {"ok": ok, "score": 1.0 if ok else 0.0, "result": result, "expected": expected}
toolkit.show({"tools": [solve_arithmetic.name, judge_result.name]})


## 2) Declarar policy y contratos antes de ejecutar

Los contratos expresan que evidencia debe aportar cada rol. La policy limita cada Agent a una llamada de Tool y mantiene la traza compacta.


In [ ]:
single_tool_policy = toolkit.RunPolicy(max_tool_calls=1, temperature=0.0, trace="compact")
solve_contract = toolkit.AgentContract(must_call=["solve_arithmetic"], tool_expectation=toolkit.expect.exactly("solve_arithmetic"), completion="when_required_tools_satisfied")
judge_contract = toolkit.AgentContract(must_call=["judge_result"], tool_expectation=toolkit.expect.exactly("judge_result"), completion="when_required_tools_satisfied")
toolkit.show({"policy": single_tool_policy.model_dump(mode="json"), "solve_contract": solve_contract.model_dump(mode="json"), "judge_contract": judge_contract.model_dump(mode="json")})


## 3) Registrar los Agents en un solo System

El System conserva ownership e inspeccion. Los Agents obligatorios usan el runtime local; el reviewer usa el runtime LM seleccionado por configuracion.


In [ ]:
@toolkit.tool
def record_review(summary: str) -> dict:
    """Registra una revision LM como evidencia estructurada."""
    return {"summary": summary}

solver = system.agent(name="deterministic_solver", instructions="Resuelve numeros estructurados.", tools=[solve_arithmetic], engine="python-runtime", runtime=local_runtime, contract=solve_contract, policy=single_tool_policy)
judge = system.agent(name="deterministic_judge", instructions="Valida resultado estructurado.", tools=[judge_result], engine="python-runtime", runtime=local_runtime, contract=judge_contract, policy=single_tool_policy)
reviewer = system.agent(name="lm_reviewer", instructions="Explica la evidencia sin cambiar numeros.", tools=[record_review], runtime=lm_runtime, policy=toolkit.RunPolicy.for_mode("eval"))
toolkit.show({
    "system": system.inspect(),
    "solver": solver.info(),
    "judge": judge.info(),
    "optional_reviewer": reviewer.info(),
}, title="Multi-agent ownership")


## 4) Ejecutar y componer el resultado end-to-end

La salida final se construye con evidencia real de solver y judge. El diagnostico LM se agrega como capacidad opcional, nunca como sustituto de la validacion.


In [ ]:
solve = solver.run({"tool": "solve_arithmetic", "input": {"numbers": NUMBERS}})
judgement = judge.run({"tool": "judge_result", "input": {"result": solve.data["result"], "expected": EXPECTED}})
review = None
review_attempt = None
review_diagnostic = {
    "status": "skipped",
    "provider": lm_resolution.get("selected_provider"),
    "reason": lm_resolution.get("reason"),
}
if lm_available:
    review_attempt = reviewer.run(str({"solution": solve.data, "judge": judgement.data}))
    if review_attempt.ok:
        review = review_attempt
        review_diagnostic = {
            "status": "ok",
            "provider": review_attempt.engine,
            "model": review_attempt.model,
        }
    else:
        review_diagnostic = {
            "status": "degraded",
            "provider": review_attempt.engine,
            "errors": review_attempt.errors,
        }
        toolkit.show(review_diagnostic, title="Optional LM review degraded")
else:
    toolkit.show(review_diagnostic, title="Optional LM review skipped")

final = {
    "procedimiento": solve.data["procedure"],
    "resultado_final": solve.data["result"],
    "judge": judgement.data,
    "lm_review": review.text if review else None,
}
system_result = toolkit.compose_result(
    text="Multi-agent system end-to-end executed.",
    data=final,
    results=[solve, judgement, review],
    mode="multi-agentic-system",
    input=USER_PROMPT,
    meta={"runtime_auto_resolution": lm_resolution, "optional_lm_review": review_diagnostic},
)
lineage = system_result.lineage(name="fundamentals.multi_agentic_system", question=USER_PROMPT, goal="Explicar solver + judge + reviewer.")
toolkit.human_result(system_result, title="Human result - Multi-Agentic System end-to-end", pretty=PRETTY, show_lineage=True, lineage=lineage)

## Lo que aprendiste

- Multi-agent significa separar responsabilidades y contratos, no duplicar Systems.
- Un mismo `AgenticSystem` puede registrar Agents deterministas y LM.
- `compose_result` preserva Tools, usage, engines y evidencia de las ejecuciones hijas.
- El judge valida el resultado de negocio; el reviewer LM solo explica evidencia.
- La degradacion de un paso opcional debe ser visible sin contaminar la ruta obligatoria.
- Este notebook no usa Graph: la secuencia solver -> judge -> reviewer es explicita.


## Coverage API de este notebook

La cobertura se concentra en ownership y composicion multiagente. Graph queda deliberadamente para el notebook 13.


In [ ]:
api_coverage = [
    "AgenticSystem",
    "AgenticSystem.agent",
    "runtime(provider='python-runtime')",
    "runtime(provider='auto')",
    "AgentContract",
    "RunPolicy",
    "compose_result",
    "RunResult.lineage",
    "human_result",
]
toolkit.show({
    "notebook": "12_multi_agentic_system_api.ipynb",
    "api_coverage": api_coverage,
})

## Simbolos API explicados

- `AgenticSystem`: owner del conjunto de Agents.
- `AgenticSystem.agent`: registra cada rol con runtime, contrato y policy.
- `AgentContract`: define la evidencia obligatoria por Agent.
- `RunPolicy`: limita el comportamiento de cada ejecucion.
- `compose_result`: consolida varios `RunResult` sin inventar evidencia.
- `RunResult.lineage`: explica la ruta multiagente y sus degradaciones.
